In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, inspect
#from sqlalchemy.exc import ProgrammingError
import pandas as pd
import gc

In [2]:
!pwd

/home/djmead/Documents/Projects/Financial/sql


In [3]:
load_dotenv()

True

In [4]:
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

In [5]:
print(f"USER: {USER}")
print(f"PASSWORD: {PASSWORD[:1]}...{PASSWORD[-1:]}")
print(f"HOST: {HOST}")
print(f"PORT: {PORT}")
print(f"DB_NAME: {DB_NAME}")

USER: postgres
PASSWORD: C...9
HOST: localhost
PORT: 5432
DB_NAME: financial_db


In [6]:
ADMIN_URL = f"postgresql+psycopg://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

admin_engine = create_engine(ADMIN_URL, isolation_level="AUTOCOMMIT")#, echo=True)

In [7]:
create_date_table_sql_path = 'CREATE_TABLE_alpaca_assets.sql'

In [8]:
def run_sql_file(file_path):
    with open(file_path, 'r') as file:
        sql = file.read()
    with admin_engine.connect() as connection:
        connection.execute(text(sql))
        #connection.commit()
    print(f"Executed SQL file: {file_path}")

In [9]:
run_sql_file(create_date_table_sql_path)

Executed SQL file: CREATE_TABLE_alpaca_assets.sql


In [11]:
def describe_table(table_name, schema="public"):
    """Return column metadata for a PostgreSQL table."""
    from sqlalchemy import inspect

    inspector = inspect(admin_engine)
    columns = inspector.get_columns(table_name, schema=schema)
    primary_key_columns = inspector.get_pk_constraint(
        table_name, schema=schema
    ).get("constrained_columns", [])

    return [
        {
            "column_name": column["name"],
            "type": str(column["type"]),
            "nullable": column["nullable"],
            "default": column["default"],
            "primary_key": column["name"] in primary_key_columns,
        }
        for column in columns
    ]


describe_table("alpaca_assets")

[{'column_name': 'id',
  'type': 'CHAR(36)',
  'nullable': False,
  'default': None,
  'primary_key': True},
 {'column_name': 'class',
  'type': 'VARCHAR(255)',
  'nullable': True,
  'default': None,
  'primary_key': False},
 {'column_name': 'exchange',
  'type': 'VARCHAR(255)',
  'nullable': True,
  'default': None,
  'primary_key': False},
 {'column_name': 'symbol',
  'type': 'VARCHAR(255)',
  'nullable': True,
  'default': None,
  'primary_key': False},
 {'column_name': 'name',
  'type': 'VARCHAR(255)',
  'nullable': True,
  'default': None,
  'primary_key': False},
 {'column_name': 'status',
  'type': 'VARCHAR(255)',
  'nullable': True,
  'default': None,
  'primary_key': False},
 {'column_name': 'tradable',
  'type': 'BOOLEAN',
  'nullable': True,
  'default': None,
  'primary_key': False},
 {'column_name': 'marginable',
  'type': 'BOOLEAN',
  'nullable': True,
  'default': None,
  'primary_key': False},
 {'column_name': 'maintenance_margin_requirement',
  'type': 'INTEGER',
  'nu